# LeetCode 1958 — Check if Move is Legal bằng Python

Link đề: https://leetcode.com/problems/check-if-move-is-legal/ · Độ khó: **Medium** · Chủ đề: Array, Enumeration, Matrix

Skeleton LeetCode:

```python
class Solution:
    def checkMove(self, board: List[List[str]], rMove: int, cMove: int, color: str) -> bool:
        ...
```

Bài mô phỏng luật kiểu cờ Othello/Reversi trên bàn `8×8`. Notebook đi từ **liệt kê mọi đoạn thẳng có đầu mút là ô vừa đi** (vét cạn) → **đi theo 8 hướng** dừng sớm → **ghép chuỗi theo hướng và so khớp mẫu** (biểu thức chính quy) → bản **tổng quát hàm `is_good_line`**.

## 1. Mục tiêu học

1. Hiểu định nghĩa **"đường tốt"**: dài ≥ 3 ô, hai đầu mút **cùng màu**, mọi ô ở giữa **màu đối phương**, không có ô trống.
2. Thành thạo kỹ thuật duyệt **8 hướng** trên lưới bằng cặp `(dr, dc)` và kiểm tra **trong biên**.
3. Biết dừng sớm đúng chỗ: gặp ô trống hoặc gặp màu mình **ngay ô kề** ⇒ hướng đó thất bại.
4. Biểu diễn một tia (ray) thành **chuỗi ký tự** và dùng **regex** `^O+C` để kiểm tra.
5. Tự sinh bàn cờ ngẫu nhiên có seed và đối chiếu nhiều cách với vét cạn.

## 2. Đề bài & ví dụ

Cho bàn cờ `board` kích thước `8 × 8` (đánh số từ 0). Ô trống là `'.'`, ô trắng `'W'`, ô đen `'B'`.

Mỗi **nước đi** chọn một ô trống và đổi nó thành màu của người chơi (`'W'` hoặc `'B'`). Nước đi **hợp lệ** chỉ khi, sau khi đổi, ô đó trở thành **đầu mút** của một **đường tốt** (ngang, dọc hoặc chéo).

**Đường tốt**: một dãy **từ 3 ô trở lên** (tính cả hai đầu mút) liên tiếp trên một đường thẳng, **hai đầu mút cùng một màu**, các ô **ở giữa** đều là **màu còn lại**, và **không có ô trống** nào.

Cho `rMove`, `cMove`, `color`, trả về `True` nếu đổi ô `(rMove, cMove)` thành `color` là nước đi hợp lệ.

**Ví dụ 1:** (bàn cờ bên dưới) `rMove = 4`, `cMove = 3`, `color = 'B'` → `True` (có hai đường tốt nhận ô này làm đầu mút).

**Ví dụ 2:** `rMove = 4`, `cMove = 4`, `color = 'W'` → `False` (có đường tốt nhận ô này làm ô **giữa**, nhưng không có đường nào nhận nó làm **đầu mút**).

**Ràng buộc:** `board` là `8 × 8`; `0 ≤ rMove, cMove < 8`; `board[rMove][cMove] == '.'`; `color` là `'B'` hoặc `'W'`.

In [1]:
import random
import re
import time
from itertools import product
from typing import List

def parse(rows):
    return [list(r) for r in rows]

B1 = parse(["...B....",
            "...W....",
            "...W....",
            "...W....",
            "WBB.WWWB",
            "...B....",
            "...B....",
            "...W...."])
B2 = parse(["........",
            ".B..W...",
            "..W.....",
            "...WB...",
            "........",
            "....BW..",
            "......W.",
            ".......B"])
examples = [(B1, 4, 3, 'B', True), (B2, 4, 4, 'W', False)]

def show(board, mark=None):
    for r in range(8):
        print(' '.join('X' if (r, c) == mark else board[r][c] for c in range(8)))

print("Ví dụ 1 (X là ô đi):"); show(B1, (4, 3))
print("\nVí dụ 2:"); show(B2, (4, 4))

Ví dụ 1 (X là ô đi):
. . . B . . . .
. . . W . . . .
. . . W . . . .
. . . W . . . .
W B B X W W W B
. . . B . . . .
. . . B . . . .
. . . W . . . .

Ví dụ 2:
. . . . . . . .
. B . . W . . .
. . W . . . . .
. . . W B . . .
. . . . X . . .
. . . . B W . .
. . . . . . W .
. . . . . . . B


## 3. Kiến thức nền

### 3.1 Tám hướng trên lưới

Mỗi hướng là một cặp `(dr, dc)` với `dr, dc ∈ {−1, 0, 1}`, bỏ `(0, 0)`. Đi `k` bước từ `(r, c)` theo hướng đó tới `(r + k·dr, c + k·dc)`.

In [2]:
DIRS = [(dr, dc) for dr in (-1, 0, 1) for dc in (-1, 0, 1) if (dr, dc) != (0, 0)]
print(len(DIRS), "hướng:", DIRS)

8 hướng: [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]


### 3.2 Kiểm tra trong biên

`0 <= r < 8 and 0 <= c < 8`. Python cho phép viết chuỗi so sánh gọn như vậy. Nếu quên, chỉ số âm `board[-1]` **không** báo lỗi mà lấy phần tử cuối — rất nguy hiểm!

In [3]:
def inside(r, c):
    return 0 <= r < 8 and 0 <= c < 8

print(inside(0, 7), inside(-1, 3), inside(8, 0))
row = list("ABCDEFGH")
print("row[-1] =", row[-1], "<- không lỗi nhưng sai ý nghĩa nếu bạn muốn 'ngoài biên'")

True False False
row[-1] = H <- không lỗi nhưng sai ý nghĩa nếu bạn muốn 'ngoài biên'


### 3.3 Màu đối phương

`opp = 'W' if color == 'B' else 'B'`. Dùng một dict cũng được: `{'B': 'W', 'W': 'B'}`.

In [4]:
OPP = {'B': 'W', 'W': 'B'}
print(OPP['B'], OPP['W'])

W B


### 3.4 Biểu thức chính quy (regex) cơ bản

`re.match(pattern, s)` khớp từ **đầu** xâu. `W+` nghĩa là "một hoặc nhiều chữ `W`". Mẫu `^W+B` khớp xâu bắt đầu bằng ≥ 1 chữ `W` rồi tới `B`.

In [5]:
for s in ["WWB.", "B", "WW.", "WWWWB", "WBW"]:
    print(f"{s!r:>8}: khớp '^W+B'? {bool(re.match(r'W+B', s))}")

  'WWB.': khớp '^W+B'? True
     'B': khớp '^W+B'? False
   'WW.': khớp '^W+B'? False
 'WWWWB': khớp '^W+B'? True
   'WBW': khớp '^W+B'? True


## 4. Ý tưởng nền tảng / trực giác

Vì ô vừa đi phải là **đầu mút**, đường tốt bắt đầu từ ô đó và đi **thẳng một hướng**. Theo hướng `(dr, dc)`, đọc các ô kế tiếp thành một chuỗi, ví dụ với ví dụ 1, ô `(4, 3)`, `color = 'B'`:
- Hướng lên `(−1, 0)`: `W W W B` ⇒ ba ô `W` rồi `B` ⇒ **tốt** (đường `B W W W B`).
- Hướng sang phải `(0, 1)`: `W W W B` ⇒ **tốt**.
- Hướng xuống `(1, 0)`: `B B W` ⇒ ô kề là màu mình ⇒ không (đường dài 2 không đủ, và không có màu đối phương ở giữa).
- Hướng sang trái `(0, −1)`: `B B W` ⇒ không.

Quy tắc theo một hướng: **≥ 1 ô màu đối phương liên tiếp**, rồi **ngay sau đó** là **một ô màu mình**.

In [6]:
def ray(board, r, c, dr, dc):
    out = []
    r, c = r + dr, c + dc
    while 0 <= r < 8 and 0 <= c < 8:
        out.append(board[r][c]); r += dr; c += dc
    return ''.join(out)

for dr, dc in DIRS:
    print(f"hướng {(dr, dc)}: {ray(B1, 4, 3, dr, dc)!r}")

hướng (-1, -1): '...'
hướng (-1, 0): 'WWWB'
hướng (-1, 1): '....'
hướng (0, -1): 'BBW'
hướng (0, 1): 'WWWB'
hướng (1, -1): '...'
hướng (1, 0): 'BBW'
hướng (1, 1): '...'


## 5. Cách 1 — Brute Force / baseline: thử mọi đoạn có đầu mút là ô vừa đi

**Ý tưởng:** đặt màu vào ô (trên **bản sao** của bàn cờ). Với mỗi hướng và mỗi độ dài `L ≥ 3` còn nằm trong bàn, lấy đoạn `L` ô bắt đầu từ ô vừa đi, kiểm tra trực tiếp theo định nghĩa đường tốt.

In [7]:
def is_good_line(cells):
    """Định nghĩa đường tốt trên một dãy ô."""
    if len(cells) < 3 or '.' in cells:
        return False
    end = cells[0]
    if cells[-1] != end:
        return False
    mid = cells[1:-1]
    other = 'W' if end == 'B' else 'B'
    return all(x == other for x in mid)

def check_move_brute(board, r, c, color):
    b = [row[:] for row in board]
    b[r][c] = color
    for dr, dc in DIRS:
        for L in range(3, 9):
            er, ec = r + (L - 1) * dr, c + (L - 1) * dc
            if not (0 <= er < 8 and 0 <= ec < 8):
                break
            cells = [b[r + k * dr][c + k * dc] for k in range(L)]
            if is_good_line(cells):
                return True
    return False

for b, r, c, col, exp in examples:
    print(check_move_brute(b, r, c, col), "(kỳ vọng", exp, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp:** 8 hướng × tối đa 6 độ dài × `O(L)` để kiểm ⇒ `O(8 · n²)` với `n = 8` — nhỏ, nhưng làm nhiều việc lặp lại (đoạn dài `L + 1` kiểm lại toàn bộ đoạn dài `L`). Bộ nhớ `O(n²)` cho bản sao.

**Nút thắt:** khi đã thấy một ô trống hoặc một ô màu mình ở giữa thì mọi đoạn dài hơn theo hướng đó **chắc chắn hỏng** — có thể dừng ngay.

## 6. Ý tưởng thuật toán / bước đột phá

Theo mỗi hướng, đi từng bước từ ô kề:
- **Nếu** ra khỏi bàn **thì** hướng này thất bại.
- **Nếu** gặp ô trống `'.'` **thì** thất bại.
- **Nếu** gặp ô màu đối phương **thì** đi tiếp (đếm `cnt += 1`).
- **Nếu** gặp ô màu mình: **nếu** `cnt ≥ 1` (đã có ít nhất một ô đối phương ở giữa) **thì** hợp lệ ⇒ trả `True`; **ngược lại** (ô kề là màu mình) thất bại.

**Bất biến:** khi đang ở bước `k` theo một hướng, mọi ô từ bước `1` tới `k − 1` đều là màu đối phương.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đi theo 8 hướng, dừng sớm

In [8]:
def check_move_walk(board, r, c, color):
    opp = 'W' if color == 'B' else 'B'
    for dr, dc in DIRS:
        i, j = r + dr, c + dc
        cnt = 0
        while 0 <= i < 8 and 0 <= j < 8 and board[i][j] == opp:
            cnt += 1
            i += dr; j += dc
        if cnt >= 1 and 0 <= i < 8 and 0 <= j < 8 and board[i][j] == color:
            return True
    return False

for b, r, c, col, exp in examples:
    print(check_move_walk(b, r, c, col), "(kỳ vọng", exp, ")")

True (kỳ vọng True )
False (kỳ vọng False )


**Độ phức tạp:** mỗi hướng đi tối đa 7 bước ⇒ `O(8 · 7)` = `O(1)` với bàn cố định (`O(n)` cho bàn `n × n`). Bộ nhớ `O(1)`; không cần sửa bàn cờ vì ô vừa đi chỉ đóng vai trò đầu mút.

### 7.2 Cách 3 — Ghép tia thành chuỗi, so khớp regex

In [9]:
def check_move_regex(board, r, c, color):
    opp = 'W' if color == 'B' else 'B'
    pattern = re.compile(f"{opp}+{color}")
    for dr, dc in DIRS:
        if pattern.match(ray(board, r, c, dr, dc)):
            return True
    return False

for b, r, c, col, exp in examples:
    print(check_move_regex(b, r, c, col), "(kỳ vọng", exp, ")")

True (kỳ vọng True )
False (kỳ vọng False )


`pattern.match` chỉ khớp **từ đầu** tia (ngay ô kề) — đúng yêu cầu "liên tiếp từ đầu mút". **Độ phức tạp:** `O(8 · n)`; regex có chi phí khởi tạo nhỏ nhưng code rất ngắn và dễ đọc.

### 7.3 Cách 4 — Dùng hàm `is_good_line` trên tiền tố ngắn nhất cần thiết

Kết hợp: đi theo hướng như cách 2 để tìm **ô đầu tiên không phải màu đối phương**, cắt đoạn từ ô đi tới ô đó, rồi kiểm bằng `is_good_line` (giúp code bám sát định nghĩa, dễ kiểm chứng).

In [10]:
def check_move_line(board, r, c, color):
    opp = 'W' if color == 'B' else 'B'
    for dr, dc in DIRS:
        cells = [color]
        i, j = r + dr, c + dc
        while 0 <= i < 8 and 0 <= j < 8:
            cells.append(board[i][j])
            if board[i][j] != opp:
                break
            i += dr; j += dc
        if is_good_line(cells):
            return True
    return False

for b, r, c, col, exp in examples:
    print(check_move_line(b, r, c, col), "(kỳ vọng", exp, ")")

True (kỳ vọng True )
False (kỳ vọng False )


Vì đường tốt phải có các ô giữa **toàn** màu đối phương, đường tốt (nếu có) theo hướng này phải kết thúc **đúng** ở ô đầu tiên khác màu đối phương — nên chỉ cần kiểm **một** đoạn mỗi hướng. **Độ phức tạp:** `O(8 · n)`.

## 8. TRACE từng bước

Trace cách 2 trên ví dụ 1 (`(4, 3)`, `color = 'B'`, `opp = 'W'`):

| hướng | các ô đi qua | cnt | ô dừng | kết luận |
|---|---|---|---|---|
| (−1,−1) | (3,2)='.' | 0 | '.' | không |
| (−1, 0) | (3,3)W (2,3)W (1,3)W | 3 | (0,3)='B' | **hợp lệ** |

In [11]:
def trace_walk(board, r, c, color):
    opp = 'W' if color == 'B' else 'B'
    for dr, dc in DIRS:
        i, j = r + dr, c + dc
        path = []
        cnt = 0
        while 0 <= i < 8 and 0 <= j < 8 and board[i][j] == opp:
            path.append(f"({i},{j}){opp}")
            cnt += 1; i += dr; j += dc
        if 0 <= i < 8 and 0 <= j < 8:
            stop = f"({i},{j})='{board[i][j]}'"
            ok = cnt >= 1 and board[i][j] == color
        else:
            stop, ok = "ngoài bàn", False
        print(f"  hướng {(dr, dc)!s:>8}: đi qua {path if path else '-'}, cnt={cnt}, dừng ở {stop} -> {'HỢP LỆ' if ok else 'không'}")
        if ok:
            return True
    return False

print("Ví dụ 1:"); print("=>", trace_walk(B1, 4, 3, 'B'))

Ví dụ 1:
  hướng (-1, -1): đi qua -, cnt=0, dừng ở (3,2)='.' -> không
  hướng  (-1, 0): đi qua ['(3,3)W', '(2,3)W', '(1,3)W'], cnt=3, dừng ở (0,3)='B' -> HỢP LỆ
=> True


In [12]:
print("Ví dụ 2:"); print("=>", trace_walk(B2, 4, 4, 'W'))

Ví dụ 2:
  hướng (-1, -1): đi qua -, cnt=0, dừng ở (3,3)='W' -> không
  hướng  (-1, 0): đi qua ['(3,4)B'], cnt=1, dừng ở (2,4)='.' -> không
  hướng  (-1, 1): đi qua -, cnt=0, dừng ở (3,5)='.' -> không
  hướng  (0, -1): đi qua -, cnt=0, dừng ở (4,3)='.' -> không
  hướng   (0, 1): đi qua -, cnt=0, dừng ở (4,5)='.' -> không
  hướng  (1, -1): đi qua -, cnt=0, dừng ở (5,3)='.' -> không
  hướng   (1, 0): đi qua ['(5,4)B'], cnt=1, dừng ở (6,4)='.' -> không
  hướng   (1, 1): đi qua -, cnt=0, dừng ở (5,5)='W' -> không
=> False


Ở ví dụ 2: hướng `(−1, −1)` đi qua `(3,3)` là `W` — **cùng màu** `W` của người chơi ngay ô kề ⇒ `cnt = 0` ⇒ không; các hướng khác gặp ô trống hoặc ra ngoài. Ô `(4, 4)` chỉ nằm **giữa** đường chéo `B(1,1) W W W(4,4) W W B(7,7)` — điều đó không được tính, vì ô vừa đi phải là **đầu mút**.

## 9. Code nộp LeetCode

In [13]:
class Solution:
    def checkMove(self, board: List[List[str]], rMove: int, cMove: int, color: str) -> bool:
        opp = 'W' if color == 'B' else 'B'
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                if dr == 0 and dc == 0:
                    continue
                i, j = rMove + dr, cMove + dc
                cnt = 0
                while 0 <= i < 8 and 0 <= j < 8 and board[i][j] == opp:
                    cnt += 1
                    i += dr
                    j += dc
                if cnt >= 1 and 0 <= i < 8 and 0 <= j < 8 and board[i][j] == color:
                    return True
        return False

for b, r, c, col, exp in examples:
    assert Solution().checkMove(b, r, c, col) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Một đường tốt nhận `(r, c)` làm đầu mút nằm trọn trên **một** hướng `d`, gồm `(r, c)`, `k ≥ 1` ô giữa và ô cuối. Vì `(r, c)` có màu `color`, ô cuối cũng phải là `color` và `k` ô giữa đều là `opp`.

Theo hướng `d`, thuật toán đi qua dãy ô liên tiếp màu `opp` **dài nhất** bắt đầu từ ô kề (dài `cnt`), rồi xét ô ngay sau.
- Nếu có đường tốt theo `d` với `k` ô giữa, thì `k` ô kề đều là `opp` và ô thứ `k + 1` là `color ≠ opp` ⇒ dãy `opp` dài nhất đúng bằng `k`, `cnt = k ≥ 1` và ô dừng là `color` ⇒ thuật toán trả `True`.
- Ngược lại, nếu thuật toán trả `True` theo `d` thì `(r, c)`, `cnt ≥ 1` ô `opp`, rồi ô `color` tạo đúng một đường tốt.
Duyệt đủ 8 hướng ⇒ đúng.

## 11. Phân tích Time / Space Complexity

Gọi `n = 8` là cạnh bàn cờ (tổng quát hóa cho bàn `n × n`).

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi đoạn | `O(8 · n²)` | `O(n²)` (bản sao) |
| 2. Đi 8 hướng | `O(8 · n)` | `O(1)` |
| 3. Chuỗi + regex | `O(8 · n)` | `O(n)` |
| 4. Một đoạn mỗi hướng + `is_good_line` | `O(8 · n)` | `O(n)` |

Với `n = 8` mọi cách đều tức thời; khác biệt chỉ có ý nghĩa khi mở rộng lên bàn lớn hoặc gọi hàng triệu lần (ví dụ AI chơi cờ).

In [14]:
rnd = random.Random(1958)
def random_board(rnd, p_empty=0.4):
    return [[rnd.choices(".BW", weights=[p_empty, (1 - p_empty) / 2, (1 - p_empty) / 2])[0] for _ in range(8)] for _ in range(8)]

boards = []
for _ in range(2000):
    b = random_board(rnd)
    empties = [(r, c) for r in range(8) for c in range(8) if b[r][c] == '.']
    if empties:
        r, c = rnd.choice(empties)
        boards.append((b, r, c, rnd.choice("BW")))
for name, fn in [("brute", check_move_brute), ("walk", check_move_walk), ("regex", check_move_regex), ("line", check_move_line), ("Solution", Solution().checkMove)]:
    t0 = time.perf_counter(); cnt = sum(fn(*x) for x in boards); dt = time.perf_counter() - t0
    print(f"{name:<9}: {cnt} nước hợp lệ / {len(boards)}  {dt * 1000:7.1f} ms")

brute    : 986 nước hợp lệ / 2000     11.2 ms
walk     : 986 nước hợp lệ / 2000      1.8 ms
regex    : 986 nước hợp lệ / 2000      5.8 ms
line     : 986 nước hợp lệ / 2000      2.7 ms
Solution : 986 nước hợp lệ / 2000      2.0 ms


## 12. Bộ test

In [15]:
ALL = {
    "walk": check_move_walk,
    "regex": check_move_regex,
    "line": check_move_line,
    "Solution": lambda b, r, c, col: Solution().checkMove(b, r, c, col),
}

def check(board, r, c, color, exp=None):
    snapshot = [row[:] for row in board]
    ref = check_move_brute(board, r, c, color)
    for name, fn in ALL.items():
        got = fn(board, r, c, color)
        assert got == ref, (name, r, c, color)
        assert board == snapshot, f"{name} đã sửa bàn cờ!"
    if exp is not None:
        assert ref == exp
    return ref

for b, r, c, col, exp in examples:
    check(b, r, c, col, exp)
print("Ví dụ đề: OK")

empty = [['.'] * 8 for _ in range(8)]
check(empty, 3, 3, 'B', False)                                   # bàn trống
b = [row[:] for row in empty]; b[0][1] = 'W'; b[0][2] = 'B'
check(b, 0, 0, 'B', True)                                        # đường ngắn nhất: B W B ở góc
b = [row[:] for row in empty]; b[0][1] = 'B'; b[0][2] = 'B'
check(b, 0, 0, 'B', False)                                       # ô kề cùng màu
b = [row[:] for row in empty]; b[0][1] = 'W'; b[0][2] = 'W'
check(b, 0, 0, 'B', False)                                       # chạm biên, không có B cuối
b = [row[:] for row in empty]
for k in range(1, 7): b[k][k] = 'W'
b[7][7] = 'B'
check(b, 0, 0, 'B', True)                                        # đường chéo dài nhất
b[3][3] = '.'
check(b, 0, 0, 'B', False)                                       # có ô trống chen giữa
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [16]:
rnd = random.Random(2010)
cnt_true = 0
for _ in range(4000):
    b = random_board(rnd, p_empty=rnd.choice([0.1, 0.3, 0.6]))
    empties = [(r, c) for r in range(8) for c in range(8) if b[r][c] == '.']
    if not empties:
        continue
    r, c = rnd.choice(empties)
    cnt_true += check(b, r, c, rnd.choice("BW"))
print(f"4000 bàn ngẫu nhiên: OK ({cnt_true} nước hợp lệ)")

4000 bàn ngẫu nhiên: OK (2298 nước hợp lệ)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chấp nhận ô kề cùng màu (đường dài 2)

In [17]:
def check_move_bug_len2(board, r, c, color):
    opp = 'W' if color == 'B' else 'B'
    for dr, dc in DIRS:
        i, j = r + dr, c + dc
        while 0 <= i < 8 and 0 <= j < 8 and board[i][j] == opp:
            i += dr; j += dc
        if 0 <= i < 8 and 0 <= j < 8 and board[i][j] == color:    # SAI: không kiểm cnt >= 1
            return True
    return False

b = [['.'] * 8 for _ in range(8)]; b[0][1] = 'B'
print("sai :", check_move_bug_len2(b, 0, 0, 'B'), "| đúng:", Solution().checkMove(b, 0, 0, 'B'))

sai : True | đúng: False


**Vì sao sai:** đường tốt cần **≥ 3 ô**, tức ít nhất **một** ô màu đối phương ở giữa. **Sửa:** đếm `cnt` và yêu cầu `cnt ≥ 1`.

### Lỗi 2 — Quên kiểm tra biên trước khi đọc ô (chỉ số âm "quay vòng")

In [18]:
def check_move_bug_bounds(board, r, c, color):
    opp = 'W' if color == 'B' else 'B'
    for dr, dc in DIRS:
        i, j = r + dr, c + dc
        cnt = 0
        steps = 0
        while board[i % 8][j % 8] == opp and steps < 8:      # SAI: không kiểm biên, dùng % cho khỏi lỗi
            cnt += 1; i += dr; j += dc; steps += 1
        if cnt >= 1 and board[i % 8][j % 8] == color:
            return True
    return False

b = [['.'] * 8 for _ in range(8)]
b[0][7] = 'W'; b[0][6] = 'B'                     # đi sang trái từ (0,0) "quay vòng" sang cột 7, 6
print("sai :", check_move_bug_bounds(b, 0, 0, 'B'), "| đúng:", Solution().checkMove(b, 0, 0, 'B'))

sai : True | đúng: False


**Vì sao sai:** trong Python, chỉ số âm (hay `% 8`) **không báo lỗi** mà đọc ô ở **phía bên kia** bàn cờ, tạo ra "đường tốt" không có thật. **Sửa:** kiểm `0 <= i < 8 and 0 <= j < 8` **trước** khi đọc `board[i][j]`.

### Lỗi 3 — Chỉ xét 4 hướng (quên đường chéo)

In [19]:
def check_move_bug_4dirs(board, r, c, color):
    opp = 'W' if color == 'B' else 'B'
    for dr, dc in [(0, 1), (0, -1), (1, 0), (-1, 0)]:          # SAI: thiếu 4 hướng chéo
        i, j = r + dr, c + dc; cnt = 0
        while 0 <= i < 8 and 0 <= j < 8 and board[i][j] == opp:
            cnt += 1; i += dr; j += dc
        if cnt >= 1 and 0 <= i < 8 and 0 <= j < 8 and board[i][j] == color:
            return True
    return False

b = [['.'] * 8 for _ in range(8)]; b[1][1] = 'W'; b[2][2] = 'B'
print("sai :", check_move_bug_4dirs(b, 0, 0, 'B'), "| đúng:", Solution().checkMove(b, 0, 0, 'B'))

sai : False | đúng: True


**Sửa:** đủ 8 hướng (gồm 4 hướng chéo).

### Lỗi 4 — Dùng `search` thay vì `match` (khớp giữa chừng) trong bản regex

In [20]:
def check_move_bug_search(board, r, c, color):
    opp = 'W' if color == 'B' else 'B'
    pattern = re.compile(f"{opp}+{color}")
    return any(pattern.search(ray(board, r, c, dr, dc)) for dr, dc in DIRS)   # SAI: search

b = [['.'] * 8 for _ in range(8)]; b[0][2] = 'W'; b[0][3] = 'B'          # (0,1) trống chen giữa
print("sai :", check_move_bug_search(b, 0, 0, 'B'), "| đúng:", Solution().checkMove(b, 0, 0, 'B'))

sai : True | đúng: False


**Vì sao sai:** `search` tìm mẫu **ở bất kỳ đâu** trên tia, kể cả sau một ô trống; đường tốt phải bắt đầu **ngay từ ô vừa đi**. **Sửa:** `match` (khớp từ đầu) hoặc thêm `^` vào mẫu.

### Lỗi 5 — Coi ô vừa đi là ô giữa (kiểm "kẹp" ở hai phía)

Trong ví dụ 2, ô `(4, 4)` nằm giữa một đường tốt. Ai kiểm "hai phía cùng màu đối phương/của mình" sẽ trả `True` sai. Đề nói rõ: ô vừa đi phải là **đầu mút**.

In [21]:
def check_move_bug_middle(board, r, c, color):
    b = [row[:] for row in board]; b[r][c] = color
    for dr, dc in DIRS:                                        # SAI: chấp nhận cả trường hợp (r,c) là ô giữa
        for L in range(3, 9):
            for start in range(L):                             # (r,c) ở vị trí bất kỳ trong đoạn
                sr, sc = r - start * dr, c - start * dc
                er, ec = sr + (L - 1) * dr, sc + (L - 1) * dc
                if all(0 <= x < 8 for x in (sr, sc, er, ec)):
                    if is_good_line([b[sr + k * dr][sc + k * dc] for k in range(L)]):
                        return True
    return False

print("ví dụ 2 — sai :", check_move_bug_middle(B2, 4, 4, 'W'), "| đúng:", Solution().checkMove(B2, 4, 4, 'W'))

ví dụ 2 — sai : True | đúng: False


**Sửa:** chỉ xét các tia **xuất phát từ** ô vừa đi.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Lưới + "đường thẳng ngang/dọc/chéo" ⇒ vòng lặp 8 hướng `(dr, dc)`.
- Luật kiểu Othello ("kẹp quân đối phương") ⇒ đi qua dãy quân đối phương dài nhất, xem ô kế tiếp.
- Kiểm biên **trước** khi truy cập — đặc biệt trong Python vì chỉ số âm hợp lệ.
- Bàn cố định `8×8` ⇒ độ phức tạp hằng số; ưu tiên code rõ ràng, ít lỗi.
- Viết hàm `is_good_line` bám sát định nghĩa để làm brute force kiểm thử.

## 15. Biến thể của bài

1. **Othello đầy đủ:** sau nước đi hợp lệ, **lật** mọi quân đối phương bị kẹp trên mọi hướng hợp lệ.
2. **Liệt kê mọi nước đi hợp lệ** cho một màu: thử mọi ô trống.
3. **Gomoku / Connect Four:** đếm chuỗi cùng màu dài nhất qua một ô theo 4 trục.
4. **LeetCode 794 — Valid Tic-Tac-Toe State**, **1275 — Find Winner on a Tic Tac Toe Game:** kiểm hàng/cột/chéo.
5. **Bàn `n × n` tổng quát:** thay số 8 bằng `n`.

In [22]:
def apply_move(board, r, c, color):
    """Biến thể 1: đặt quân và lật mọi quân bị kẹp; trả về bàn mới (None nếu không hợp lệ)."""
    if not Solution().checkMove(board, r, c, color):
        return None
    opp = 'W' if color == 'B' else 'B'
    b = [row[:] for row in board]; b[r][c] = color
    for dr, dc in DIRS:
        i, j = r + dr, c + dc; path = []
        while 0 <= i < 8 and 0 <= j < 8 and board[i][j] == opp:
            path.append((i, j)); i += dr; j += dc
        if path and 0 <= i < 8 and 0 <= j < 8 and board[i][j] == color:
            for x, y in path:
                b[x][y] = color
    return b

def legal_moves(board, color):
    return [(r, c) for r in range(8) for c in range(8) if board[r][c] == '.' and Solution().checkMove(board, r, c, color)]

after = apply_move(B1, 4, 3, 'B')
print("Ví dụ 1 sau khi đi và lật quân:"); show(after)
print("Các nước hợp lệ của B trên ví dụ 1:", legal_moves(B1, 'B'))

Ví dụ 1 sau khi đi và lật quân:
. . . B . . . .
. . . B . . . .
. . . B . . . .
. . . B . . . .
W B B B B B B B
. . . B . . . .
. . . B . . . .
. . . W . . . .
Các nước hợp lệ của B trên ví dụ 1: [(2, 4), (3, 5), (4, 3)]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi đoạn | đặt quân, kiểm mọi độ dài | `O(8n²)` | `O(n²)` | đối chiếu |
| 2. Đi 8 hướng | dãy `opp` rồi ô `color` | `O(8n)` | `O(1)` | **nộp bài** |
| 3. Regex | tia thành chuỗi, `^opp+color` | `O(8n)` | `O(n)` | code ngắn |
| 4. `is_good_line` | một đoạn mỗi hướng | `O(8n)` | `O(n)` | bám định nghĩa |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Trên bàn trống chỉ có `board[2][2] = 'W'`, `board[3][3] = 'B'`; đi `(1, 1)` màu `'B'` có hợp lệ không? Còn màu `'W'`?

**Bài 2 (tính tay):** Với ví dụ 1, liệt kê đủ 8 hướng và cho biết hướng nào cho đường tốt khi `color = 'B'` tại `(4, 3)`.

**Bài 3 (điền TODO):** Hoàn thiện `good_directions(board, r, c, color)` trả về **danh sách các hướng** `(dr, dc)` cho đường tốt.

**Bài 4 (tự viết lại):** Viết `check_move_n(board, r, c, color)` cho bàn **`n × n` bất kỳ** (không giả định 8).

**Bài 5 (mini challenge):** Viết `count_flips(board, r, c, color)` trả về **số quân đối phương bị lật** nếu đi nước đó (0 nếu không hợp lệ), và `best_move(board, color)` chọn nước lật nhiều nhất (hòa thì chọn ô nhỏ nhất theo `(r, c)`).

In [23]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def good_directions(board, r, c, color):
    # TODO: như Solution nhưng gom mọi (dr, dc) thỏa thay vì return True
    pass

# Bài 4 — TODO
def check_move_n(board, r, c, color):
    # TODO: n = len(board), m = len(board[0]); thay 8 bằng n, m
    pass

# Bài 5 — TODO
def count_flips(board, r, c, color):
    # TODO: cộng cnt của mọi hướng hợp lệ
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Hướng `(1, 1)` từ `(1, 1)`: gặp `W` rồi `B`.
- Mức 2: Với `'B'`: `W` là đối phương, `B` là màu mình ⇒ hợp lệ.
- Mức 3: Với `'W'`: ô kề là màu mình ⇒ hướng đó không được.

**Bài 2**
- Mức 1: Dùng `ray()` in 8 tia.
- Mức 2: Chuỗi phải khớp `W+B`.
- Mức 3: Hai hướng `(−1, 0)` và `(0, 1)`.

**Bài 3**
- Mức 1: Thay `return True` bằng `res.append((dr, dc))`.
- Mức 2: Kiểm kết quả rỗng ⇔ `Solution()` trả `False`.
- Mức 3: So với brute force theo từng hướng.

**Bài 4**
- Mức 1: Kích thước lấy từ `len(board)`.
- Mức 2: Bàn có thể chữ nhật `n × m`.
- Mức 3: Với bàn `8×8` phải khớp `Solution`.

**Bài 5**
- Mức 1: Một hướng hợp lệ lật đúng `cnt` quân.
- Mức 2: Các hướng không chồng nhau nên cộng được.
- Mức 3: `best_move` duyệt mọi ô trống.

## 19. Đáp án tham khảo cho bài tập

In [24]:
# Bài 1
b = [['.'] * 8 for _ in range(8)]; b[2][2] = 'W'; b[3][3] = 'B'
r1 = (Solution().checkMove(b, 1, 1, 'B'), Solution().checkMove(b, 1, 1, 'W'))
print("Bài 1:", r1)
assert r1 == (True, False)

# Bài 2 & 3
def good_directions_sol(board, r, c, color):
    opp = 'W' if color == 'B' else 'B'
    res = []
    for dr, dc in DIRS:
        i, j, cnt = r + dr, c + dc, 0
        while 0 <= i < 8 and 0 <= j < 8 and board[i][j] == opp:
            cnt += 1; i += dr; j += dc
        if cnt and 0 <= i < 8 and 0 <= j < 8 and board[i][j] == color:
            res.append((dr, dc))
    return res

print("Bài 2/3:", good_directions_sol(B1, 4, 3, 'B'))
assert good_directions_sol(B1, 4, 3, 'B') == [(-1, 0), (0, 1)]
rnd = random.Random(3)
for _ in range(1000):
    bb = random_board(rnd)
    em = [(x, y) for x in range(8) for y in range(8) if bb[x][y] == '.']
    if em:
        x, y = rnd.choice(em); col = rnd.choice("BW")
        assert bool(good_directions_sol(bb, x, y, col)) == Solution().checkMove(bb, x, y, col)

Bài 1: (True, False)
Bài 2/3: [(-1, 0), (0, 1)]


In [25]:
# Bài 4
def check_move_n_sol(board, r, c, color):
    n, m = len(board), len(board[0])
    opp = 'W' if color == 'B' else 'B'
    for dr, dc in DIRS:
        i, j, cnt = r + dr, c + dc, 0
        while 0 <= i < n and 0 <= j < m and board[i][j] == opp:
            cnt += 1; i += dr; j += dc
        if cnt and 0 <= i < n and 0 <= j < m and board[i][j] == color:
            return True
    return False

rnd = random.Random(4)
for _ in range(1000):
    bb = random_board(rnd)
    em = [(x, y) for x in range(8) for y in range(8) if bb[x][y] == '.']
    if em:
        x, y = rnd.choice(em); col = rnd.choice("BW")
        assert check_move_n_sol(bb, x, y, col) == Solution().checkMove(bb, x, y, col)
small = [list(".WB"), list("..."), list("...")]
print("Bài 4: bàn 3x3, đi (0,0) màu B:", check_move_n_sol(small, 0, 0, 'B'))

# Bài 5
def count_flips_sol(board, r, c, color):
    opp = 'W' if color == 'B' else 'B'
    total = 0
    for dr, dc in DIRS:
        i, j, cnt = r + dr, c + dc, 0
        while 0 <= i < 8 and 0 <= j < 8 and board[i][j] == opp:
            cnt += 1; i += dr; j += dc
        if cnt and 0 <= i < 8 and 0 <= j < 8 and board[i][j] == color:
            total += cnt
    return total

def best_move_sol(board, color):
    best = None
    for r in range(8):
        for c in range(8):
            if board[r][c] == '.':
                f = count_flips_sol(board, r, c, color)
                if f and (best is None or f > best[0]):
                    best = (f, (r, c))
    return best

for _ in range(500):
    bb = random_board(rnd)
    em = [(x, y) for x in range(8) for y in range(8) if bb[x][y] == '.']
    if em:
        x, y = rnd.choice(em); col = rnd.choice("BW")
        after = apply_move(bb, x, y, col)
        flips = count_flips_sol(bb, x, y, col)
        if after is None:
            assert flips == 0
        else:
            assert flips == sum(bb[i][j] != after[i][j] for i in range(8) for j in range(8)) - 1
print("Bài 5 OK | ví dụ 1: lật", count_flips_sol(B1, 4, 3, 'B'), "quân; nước tốt nhất của B:", best_move_sol(B1, 'B'))

Bài 4: bàn 3x3, đi (0,0) màu B: True
Bài 5 OK | ví dụ 1: lật 6 quân; nước tốt nhất của B: (6, (4, 3))


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đi theo 8 hướng kiểu Othello":**

```python
opp = 'W' if color == 'B' else 'B'
for dr in (-1, 0, 1):
    for dc in (-1, 0, 1):
        if dr == dc == 0: continue
        i, j, cnt = r + dr, c + dc, 0
        while 0 <= i < N and 0 <= j < N and board[i][j] == opp:
            cnt += 1; i += dr; j += dc
        if cnt >= 1 and 0 <= i < N and 0 <= j < N and board[i][j] == color:
            return True
return False
```

**Checklist:**
- [ ] Đủ **8** hướng (gồm chéo)?
- [ ] Cần **≥ 1** ô đối phương ở giữa (đường ≥ 3 ô)?
- [ ] Kiểm biên **trước** khi đọc ô (tránh chỉ số âm)?
- [ ] Ô vừa đi là **đầu mút**, không phải ô giữa?
- [ ] Không sửa `board` đầu vào?
- [ ] Đã so với brute force trên bàn ngẫu nhiên?
- [ ] Chữ ký đúng: `def checkMove(self, board: List[List[str]], rMove: int, cMove: int, color: str) -> bool`?